# Program 10: Sentiment analysis on user-entered sentences

**Aim.** Load the sentiment pipeline and analyse sentences supplied by the user, handling uncertainty.

**Theory.** Real input is messy: negation, sarcasm, mixed opinions. The program reports a third answer, UNSURE, when the model's confidence is below a threshold, and is scored on a small labelled set so its accuracy is a number, not a feeling.

In [1]:
import sys
sys.path.insert(0, '.')
from common import *


In [2]:
from transformers import pipeline
clf = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')

def analyse(text, threshold=0.90):
    r = clf(text)[0]
    return ('UNSURE' if r['score'] < threshold else r['label']), round(r['score'], 3)

# interactive use: print(analyse(input('Sentence: ')))
for s in ['I love how simple this plan is.', 'This is not good at all.', 'Not bad.', 'Oh great, another pill to remember.', 'The tablet is white and round.']:
    print(analyse(s), '<-', s)

('POSITIVE', 1.0) <- I love how simple this plan is.
('NEGATIVE', 1.0) <- This is not good at all.
('POSITIVE', 0.999) <- Not bad.
('POSITIVE', 0.997) <- Oh great, another pill to remember.
('UNSURE', 0.794) <- The tablet is white and round.


In [3]:
labelled = [('Excellent service and clear advice.', 'POSITIVE'), ('Terrible, I was ignored.', 'NEGATIVE'), ('The app is wonderful.', 'POSITIVE'),
  ('I am very unhappy with the delay.', 'NEGATIVE'), ('Helpful and kind staff.', 'POSITIVE'), ('The instructions were awful.', 'NEGATIVE'),
  ('I would recommend this pharmacy.', 'POSITIVE'), ('It made my pain worse and nobody helped.', 'NEGATIVE'), ('Very reassuring explanation.', 'POSITIVE'),
  ('A waste of my time.', 'NEGATIVE'), ('Quick and painless process.', 'POSITIVE'), ('Rude and dismissive.', 'NEGATIVE')]
pred = [clf(t)[0]['label'] for t, _ in labelled]
acc = sum(p == y for p, (_, y) in zip(pred, labelled)) / len(labelled)
print(f'accuracy on {len(labelled)} hand-written sentences: {acc:.0%}  (a tiny set: it shows the method, not the true accuracy)')

accuracy on 12 hand-written sentences: 92%  (a tiny set: it shows the method, not the true accuracy)


## Observations

The UNSURE band catches some hard cases, not all (sarcasm can still score high). A 12-sentence test cannot give a reliable accuracy; it demonstrates how to measure.